# 6.0 Pre-Procesing

# 6.1 Import library

In [3]:
import pandas as pd
import numpy as np
import re
import string
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pd.set_option("display.max_colwidth", 200)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Load Dataset

In [4]:
file_path = "archive\Combined Data.csv"

df = pd.read_csv(file_path)

print("Original dataset shape:", df.shape)
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\irfan\AppData\Local\Temp\ipykernel_50412\1502534058.py:1: SyntaxWarning: invalid escape sequence '\C'
  file_path = "archive\Combined Data.csv"


Original dataset shape: (53043, 3)


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless heart. All out of tune",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay in a restless and restless place",Anxiety
3,3,I've shifted my focus to something else but I'm still worried,Anxiety
4,4,"I'm restless and restless, it's been a month now, boy. What do you mean?",Anxiety


## Standardise column names

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Column names:")
print(df.columns.tolist())

Column names:
['unnamed:_0', 'statement', 'status']


## Drop irrelevant index columns

In [6]:
# Drop columns that are only index/identifier columns
columns_to_drop = [col for col in df.columns if "unnamed" in col]

df = df.drop(columns=columns_to_drop)

print("Shape after dropping irrelevant columns:", df.shape)
df.head()

Shape after dropping irrelevant columns: (53043, 2)


,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless heart. All out of tune",Anxiety
2,"All wrong, back off dear, forward doubt. Stay in a restless and restless place",Anxiety
3,I've shifted my focus to something else but I'm still worried,Anxiety
4,"I'm restless and restless, it's been a month now, boy. What do you mean?",Anxiety


## Select relevant columns only

In [7]:
df_selected = df[["statement", "status"]].copy()

df_selected = df_selected.rename(columns={
    "statement": "text",
    "status": "label"
})

print("Selected dataset shape:", df_selected.shape)
df_selected.head()

Selected dataset shape: (53043, 2)


,text,label
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless heart. All out of tune",Anxiety
2,"All wrong, back off dear, forward doubt. Stay in a restless and restless place",Anxiety
3,I've shifted my focus to something else but I'm still worried,Anxiety
4,"I'm restless and restless, it's been a month now, boy. What do you mean?",Anxiety


## Check before preprocessing

In [8]:
before_preprocessing_summary = pd.DataFrame({
    "Item": [
        "Rows before preprocessing",
        "Missing text values",
        "Missing label values",
        "Duplicate text statements",
        "Number of classes"
    ],
    "Value": [
        df_selected.shape[0],
        df_selected["text"].isnull().sum(),
        df_selected["label"].isnull().sum(),
        df_selected.duplicated(subset=["text"]).sum(),
        df_selected["label"].nunique()
    ]
})

before_preprocessing_summary

,Item,Value
0,Rows before preprocessing,53043
1,Missing text values,362
2,Missing label values,0
3,Duplicate text statements,1969
4,Number of classes,7


## Remove missing values

In [9]:
before_missing_removal = df_selected.shape[0]

df_preprocessed = df_selected.dropna(subset=["text", "label"]).copy()

after_missing_removal = df_preprocessed.shape[0]

print("Rows before missing value removal:", before_missing_removal)
print("Rows after missing value removal:", after_missing_removal)
print("Rows removed due to missing values:", before_missing_removal - after_missing_removal)

Rows before missing value removal: 53043
Rows after missing value removal: 52681
Rows removed due to missing values: 362


## Remove duplicate statements

In [10]:
before_duplicate_removal = df_preprocessed.shape[0]

df_preprocessed = df_preprocessed.drop_duplicates(subset=["text"]).copy()

after_duplicate_removal = df_preprocessed.shape[0]

print("Rows before duplicate removal:", before_duplicate_removal)
print("Rows after duplicate removal:", after_duplicate_removal)
print("Duplicate statements removed:", before_duplicate_removal - after_duplicate_removal)

Rows before duplicate removal: 52681
Rows after duplicate removal: 51073
Duplicate statements removed: 1608


## Clean text

In [11]:
def clean_text_lighter(text):
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Remove user mentions
    text = re.sub(r"@\w+", " ", text)
    
    # Keep hashtag word but remove symbol
    text = re.sub(r"#", "", text)
    
    # Convert emotional punctuation into tokens
    text = re.sub(r"!+", " exclamation ", text)
    text = re.sub(r"\?+", " question ", text)
    
    # Expand common negation contractions
    text = re.sub(r"can't", "can not", text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"n't", " not", text)
    
    # Remove remaining punctuation except apostrophes handled above
    text = text.translate(str.maketrans("", "", string.punctuation))
    
    # Keep words and numbers
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [12]:
df_preprocessed["clean_text"] = df_preprocessed["text"].apply(clean_text_lighter)

df_preprocessed[["text", "clean_text", "label"]].head(10)

,text,clean_text,label
0,oh my gosh,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless heart. All out of tune",trouble sleeping confused mind restless heart all out of tune,Anxiety
2,"All wrong, back off dear, forward doubt. Stay in a restless and restless place",all wrong back off dear forward doubt stay in a restless and restless place,Anxiety
3,I've shifted my focus to something else but I'm still worried,ive shifted my focus to something else but im still worried,Anxiety
4,"I'm restless and restless, it's been a month now, boy. What do you mean?",im restless and restless its been a month now boy what do you mean question,Anxiety
5,"every break, you must be nervous, like something is wrong, but what the heck",every break you must be nervous like something is wrong but what the heck,Anxiety
6,"I feel scared, anxious, what can I do? And may my family or us be protected :)",i feel scared anxious what can i do question and may my family or us be protected,Anxiety
7,Have you ever felt nervous but didn't know why?,have you ever felt nervous but did not know why question,Anxiety
8,"I haven't slept well for 2 days, it's like I'm restless. why huh :([].",i have not slept well for 2 days its like im restless why huh,Anxiety
9,"I'm really worried, I want to cry.",im really worried i want to cry,Anxiety


## Remove empty text after cleaning

In [13]:
before_empty_removal = df_preprocessed.shape[0]

df_preprocessed["clean_text"] = df_preprocessed["clean_text"].replace("", np.nan)
df_preprocessed = df_preprocessed.dropna(subset=["clean_text"]).copy()

after_empty_removal = df_preprocessed.shape[0]

print("Rows before empty text removal:", before_empty_removal)
print("Rows after empty text removal:", after_empty_removal)
print("Rows removed due to empty cleaned text:", before_empty_removal - after_empty_removal)

Rows before empty text removal: 51073
Rows after empty text removal: 51066
Rows removed due to empty cleaned text: 7


## Check class distribution after preprocessing

In [14]:
class_distribution_after = df_preprocessed["label"].value_counts().reset_index()
class_distribution_after.columns = ["Mental Health Status", "Count"]
class_distribution_after["Percentage"] = (
    class_distribution_after["Count"] / class_distribution_after["Count"].sum() * 100
).round(2)

class_distribution_after

,Mental Health Status,Count,Percentage
0,Normal,16033,31.40
1,Depression,15087,29.54
2,Suicidal,10640,20.84
3,Anxiety,3617,7.08
4,Bipolar,2501,4.90
5,Stress,2293,4.49
6,Personality disorder,895,1.75


## Encode labels

In [15]:
label_encoder = LabelEncoder()

df_preprocessed["encoded_label"] = label_encoder.fit_transform(df_preprocessed["label"])

label_mapping = pd.DataFrame({
    "Original Label": label_encoder.classes_,
    "Encoded Label": range(len(label_encoder.classes_))
})

label_mapping

,Original Label,Encoded Label
0,Anxiety,0
1,Bipolar,1
2,Depression,2
3,Normal,3
4,Personality disorder,4
5,Stress,5
6,Suicidal,6


## Prepare X and y

In [16]:
X = df_preprocessed["clean_text"].values
y = df_preprocessed["encoded_label"].values

print("Number of text samples:", len(X))
print("Number of labels:", len(y))
print("Number of classes:", len(label_encoder.classes_))

Number of text samples: 51066
Number of labels: 51066
Number of classes: 7


## Train-test split

In [17]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

Training samples: 40852
Testing samples: 10214


In [18]:
train_distribution = pd.Series(y_train).value_counts().sort_index()
test_distribution = pd.Series(y_test).value_counts().sort_index()

split_distribution = pd.DataFrame({
    "Class Label": label_encoder.classes_,
    "Train Count": train_distribution.values,
    "Test Count": test_distribution.values
})

split_distribution

,Class Label,Train Count,Test Count
0,Anxiety,2894,723
1,Bipolar,2001,500
2,Depression,12069,3018
3,Normal,12826,3207
4,Personality disorder,716,179
5,Stress,1834,459
6,Suicidal,8512,2128


## Decide tokenizer settings

| EDA result                       | Decision                                  |
| -------------------------------- | ----------------------------------------- |
| Median word length = 61          | Many statements are moderate length       |
| 75th percentile = 148            | Good coverage if max length is around 150 |
| 90th percentile = 277            | Very long statements exist                |
| 95th percentile = 394            | Some statements are extremely long        |
| Total unique vocabulary = 77,113 | Use vocabulary limit                      |


In [19]:
max_words = 20000
max_len = 150

print("Maximum vocabulary size:", max_words)
print("Maximum sequence length:", max_len)

Maximum vocabulary size: 20000
Maximum sequence length: 150


Why not 394 or 6300? Because very long sequences increase training time heavily. A max_len of 150 covers around 75% of statements fully and truncates very long texts to keep training practical.

## Tokenization

In [20]:
tokenizer = Tokenizer(
    num_words=max_words,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

print("Example original text:")
print(X_train_text[0])

print("\nExample token sequence:")
print(X_train_seq[0][:30])

Example original text:
joey jordison former drummer of slipknot just died at age 46 and i am in nothing but shock because i did not expect to see this in another 20 years at the very least he went out peacefully in his sleep so i am glad for that joey jordison

Example token sequence:
[16930, 1, 2727, 16931, 10, 16932, 20, 549, 36, 486, 6481, 4, 2, 15, 16, 125, 18, 2547, 40, 2, 82, 8, 986, 3, 121, 21, 16, 262, 677, 95]


In [21]:
word_index = tokenizer.word_index

print("Total words found by tokenizer:", len(word_index))
print("First 20 words in tokenizer:")
list(word_index.items())[:20]

Total words found by tokenizer: 66279
First 20 words in tokenizer:


[('<OOV>', 1),
 ('i', 2),
 ('to', 3),
 ('and', 4),
 ('the', 5),
 ('my', 6),
 ('a', 7),
 ('not', 8),
 ('it', 9),
 ('of', 10),
 ('is', 11),
 ('me', 12),
 ('have', 13),
 ('that', 14),
 ('am', 15),
 ('in', 16),
 ('do', 17),
 ('but', 18),
 ('for', 19),
 ('just', 20)]

## Padding

In [22]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

print("Training padded shape:", X_train_pad.shape)
print("Testing padded shape:", X_test_pad.shape)

Training padded shape: (40852, 150)
Testing padded shape: (10214, 150)


In [24]:
print("Example padded sequence:")
print(X_train_pad[0])

Example padded sequence:
[16930     1  2727 16931    10 16932    20   549    36   486  6481     4
     2    15    16   125    18  2547    40     2    82     8   986     3
   121    21    16   262   677    95    36     5   136   323    65   216
    45  3574    16   208   206    22     2    15  1493    19    14 16930
     1     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0]


## Final preprocessing summary

In [25]:
preprocessing_summary = pd.DataFrame({
    "Step": [
        "Original dataset rows",
        "Rows after removing missing/corrupted/duplicate/empty text",
        "Number of classes",
        "Training samples",
        "Testing samples",
        "Maximum vocabulary size",
        "Maximum sequence length",
        "Input shape for LSTM/BiLSTM"
    ],
    "Result": [
        df.shape[0],
        df_preprocessed.shape[0],
        len(label_encoder.classes_),
        X_train_pad.shape[0],
        X_test_pad.shape[0],
        max_words,
        max_len,
        str(X_train_pad.shape)
    ]
})

preprocessing_summary

,Step,Result
0,Original dataset rows,53043
1,Rows after removing missing/corrupted/duplicate/empty text,51066
2,Number of classes,7
3,Training samples,40852
4,Testing samples,10214
5,Maximum vocabulary size,20000
6,Maximum sequence length,150
7,Input shape for LSTM/BiLSTM,"(40852, 150)"


## Save preprocessed dataset

In [29]:
df_preprocessed.to_csv("preprocessed_mental_health_dataset.csv", index=False)

print("Preprocessed dataset saved as preprocessed_mental_health_dataset.csv")

Preprocessed dataset saved as preprocessed_mental_health_dataset.csv


## Save train/test arrays for Assignment 2

In [30]:
np.save("X_train_pad.npy", X_train_pad)
np.save("X_test_pad.npy", X_test_pad)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

print("Train/test arrays saved successfully.")

Train/test arrays saved successfully.


## Save tokenizer and label encoder

In [31]:
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Tokenizer and label encoder saved successfully.")

Tokenizer and label encoder saved successfully.
